# Notebook 3 — Predictions and Refusals
**YTS+ DSEB 2026 · Project A · Harry Potter**

---

## The question for this session

Two characters who share many mutual acquaintances inhabit the same narrative world. Rowling tends to eventually put such characters in a scene together.

We can measure this — **common neighbors** — and use it to predict which characters Rowling will put together. After Book 3, which pairs are most likely to meet?

And then: which pairs share many mutual characters, are both alive, are both still in the story — and Rowling **never** put them in the same scene?

Those refusals are deliberate. **Which one would you write into the movies?**

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'
print('Ready.')

---
## Part 1 — EDA: Common neighbors after Book 3

The pre-computed file contains every pair of characters with at least 3 common neighbors after Book 3. For each pair: how many mutual characters they share, whether they ever meet across the full series, and in which book they first meet.

In [ ]:
cn = pd.read_csv(DATA + 'common_neighbors_book3.csv')

print(f'Shape: {cn.shape}')
print(f'Columns: {cn.columns.tolist()}')
print()
cn.head(10)

In [ ]:
# Distribution of common neighbor counts
print('Common neighbor count distribution:')
print(cn['common_neighbors'].describe())
print()
print(f'Pairs with CN >= 5:  {(cn["common_neighbors"] >= 5).sum()}')
print(f'Pairs with CN >= 10: {(cn["common_neighbors"] >= 10).sum()}')
print(f'Pairs with CN >= 15: {(cn["common_neighbors"] >= 15).sum()}')
print(f'Pairs with CN >= 20: {(cn["common_neighbors"] >= 20).sum()}')

In [ ]:
# Of all pairs with CN >= 3, how many eventually meet?
total = len(cn)
ever_meet = cn['ever_meet'].sum()
never_meet = total - ever_meet

print(f'Total pairs (CN >= 3): {total}')
print(f'  Meet somewhere in Books 1-7: {ever_meet} ({ever_meet/total*100:.0f}%)')
print(f'  Never meet:                  {never_meet} ({never_meet/total*100:.0f}%)')
print()

# How does this rate change as CN increases?
print('Meet rate by CN threshold:')
for threshold in [5, 10, 15, 20]:
    sub = cn[cn['common_neighbors'] >= threshold]
    rate = sub['ever_meet'].mean()
    print(f'  CN >= {threshold:>2}: {len(sub):>5} pairs | {rate*100:.0f}% eventually meet')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

meet = cn[cn['ever_meet'] == True]['common_neighbors']
no_meet = cn[cn['ever_meet'] == False]['common_neighbors']

ax.hist(no_meet, bins=30, color='tomato', alpha=0.7, label='Never meet', edgecolor='white')
ax.hist(meet, bins=30, color='steelblue', alpha=0.7, label='Eventually meet', edgecolor='white')
ax.set_xlabel('Common neighbors after Book 3')
ax.set_ylabel('Number of pairs')
ax.set_title('Common neighbors after Book 3 — do they eventually meet?')
ax.legend()
plt.tight_layout()
plt.savefig('images/cn_distribution.png', dpi=150)
plt.show()

In [ ]:
# For pairs that do meet — when do they first meet?
meet_pairs = cn[cn['ever_meet'] == True].copy()
book_counts = meet_pairs['first_meet_book'].value_counts().sort_index()

print('When do high-CN pairs first meet?')
print(book_counts.to_string())

---
## Part 2 — EDA: Never-meeting pairs

In [ ]:
nm = pd.read_csv(DATA + 'never_meeting_pairs.csv')

print(f'Shape: {nm.shape}')
print(f'Columns: {nm.columns.tolist()}')
print()
nm.head(8)

In [ ]:
# How many of these are genuine refusals vs explained by death/exit?
print('Never-meeting pairs breakdown:')
print(f'  Total pairs (CN >= 15 after Book 5): {len(nm)}')
print(f'  Dead character:       {nm["is_dead_character"].sum()}')
print(f'  Historical figure:    {nm["is_historical_figure"].sum()}')
print(f'  Narrative exit:       {nm["is_narrative_exit"].sum()}')
print(f'  Genuine refusals:     {nm["genuine_refusal"].sum()}')
print()

genuine = nm[nm['genuine_refusal'] == True].sort_values(
    'common_neighbors_after_book5', ascending=False)
print(f'Top genuine refusals:')
print(genuine[['char_a', 'char_b', 'common_neighbors_after_book5']].head(10).to_string(index=False))

In [ ]:
# CN distribution: genuine refusals vs explained absences
genuine = nm[nm['genuine_refusal'] == True]
explained = nm[nm['genuine_refusal'] == False]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(explained['common_neighbors_after_book5'], bins=20, color='gray', alpha=0.6,
        label='Explained (dead / exited)', edgecolor='white')
ax.hist(genuine['common_neighbors_after_book5'], bins=20, color='firebrick', alpha=0.7,
        label='Genuine refusal', edgecolor='white')
ax.set_xlabel('Common neighbors after Book 5')
ax.set_ylabel('Number of pairs')
ax.set_title('Never-meeting pairs — CN after Book 5')
ax.legend()
plt.tight_layout()
plt.savefig('images/genuine_refusals_cn.png', dpi=150)
plt.show()

---
## Part 3 — Your predictions

**Write down now — before searching the data:**
- 3 pairs you predict will meet in Book 4 (haven't yet after Book 3)
- 3 pairs you predict will NEVER meet across all 7 books

**My predictions:**

Will meet in Book 4:
1. 
2. 
3. 

Will never meet:
1. 
2. 
3. 

In [ ]:
# Look up any pair in the common neighbors table
# Change these to your predicted pairs

def lookup_pair(a, b):
    match = cn[((cn['char_a'] == a) & (cn['char_b'] == b)) |
               ((cn['char_a'] == b) & (cn['char_b'] == a))]
    if len(match) == 0:
        print(f'{a} / {b}: CN < 3 (barely share mutual characters)')
    else:
        row = match.iloc[0]
        print(f'{a} / {b}:')
        print(f'  Common neighbors: {int(row["common_neighbors"])}')
        print(f'  Ever meet: {row["ever_meet"]}')
        if row['ever_meet']:
            print(f'  First meet: Book {int(row["first_meet_book"])}')

# Replace with your pairs
lookup_pair('Harry Potter', 'Draco Malfoy')    # example — they obviously meet
print()
lookup_pair('Gilderoy Lockhart', 'Sirius Black')  # known interesting failure

**Examine the failures.** The network predicted a meeting that never happened.

For each failure, which category explains it?
- Character died or was obliviated (exits the story)
- Character graduated or left Hogwarts
- Historical figure (not a real scene character)
- Rowling made a deliberate choice to keep them apart

That last category is the interesting one.

---
## Part 4 — How good is the prediction?

In [ ]:
# Precision at different CN thresholds
# Precision = of pairs predicted to meet (CN >= threshold), how many actually did?

print(f'{"CN threshold":<15} {"Pairs predicted":>16} {"Actually met":>13} {"Precision":>10}')
print('-' * 58)

for threshold in [5, 8, 10, 12, 15, 20]:
    predicted = cn[cn['common_neighbors'] >= threshold]
    met = predicted['ever_meet'].sum()
    precision = met / len(predicted) if len(predicted) > 0 else 0
    print(f'CN >= {threshold:<9} {len(predicted):>16} {int(met):>13} {precision*100:>9.0f}%')

In [ ]:
thresholds = list(range(3, 25))
precisions = []
pair_counts = []

for t in thresholds:
    sub = cn[cn['common_neighbors'] >= t]
    precisions.append(sub['ever_meet'].mean() if len(sub) > 0 else 0)
    pair_counts.append(len(sub))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(thresholds, [p*100 for p in precisions], 'o-', color='steelblue', linewidth=2, label='Precision (%)')
ax2.bar(thresholds, pair_counts, alpha=0.2, color='gray', label='Pairs predicted')

ax1.axhline(92, color='steelblue', linestyle='--', alpha=0.5)
ax1.text(22, 93, '92%', color='steelblue', fontsize=9)

ax1.set_xlabel('Common neighbor threshold')
ax1.set_ylabel('Precision (% who actually meet)', color='steelblue')
ax2.set_ylabel('Number of pairs predicted', color='gray')
ax1.set_title('Link prediction — precision vs threshold')
plt.tight_layout()
plt.savefig('images/precision_curve.png', dpi=150)
plt.show()

**The rule:** CN > 10 after Book 3 → 92% chance they meet before Book 7 ends.

As the threshold goes up, precision goes up but the number of pairs you're predicting goes down. At CN >= 15, precision is very high — but you're only predicting a handful of pairs.

**Write:** Is a 92% precision good or bad? What does the 8% failure tell you about Rowling's narrative choices?

**Your answer:**

*Write here.*

---
## Part 5 — The genuine refusals

In [ ]:
genuine = nm[nm['genuine_refusal'] == True].sort_values(
    'common_neighbors_after_book5', ascending=False).reset_index(drop=True)

print(f'Genuine refusals: {len(genuine)} pairs\n')
print(genuine[['char_a', 'char_b', 'common_neighbors_after_book5']].head(12).to_string(index=False))

In [ ]:
top8 = genuine.head(8)
labels = [f"{row['char_a'].split()[-1]} –\n{row['char_b'].split()[-1]}"
          for _, row in top8.iterrows()]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels[::-1], top8['common_neighbors_after_book5'].tolist()[::-1], color='firebrick')
ax.set_xlabel('Common neighbors after Book 5')
ax.set_title('Genuine refusals — high-CN pairs who never share a scene')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('images/genuine_refusals_bar.png', dpi=150)
plt.show()

**For each top refusal, think about why Rowling kept them apart.**

- McGonagall — Cho Chang: CN=40. Both deeply embedded in Hogwarts. Never share a scene.
- Fudge — Ginny: CN=36. Minister of Magic and a prominent Weasley. Never share a scene.
- Filch — Tom Riddle: CN=31. The caretaker and the Dark Lord. The mundane and the supernatural kept completely separate.
- Fred Weasley — Seamus Finnigan: CN=31. Both Gryffindors. Never explicitly share a scene.

These are not accidents. They reveal how Rowling organised her story worlds.

---
## Part 6 — The screenwriter question

You are adapting Books 5–7 into films. You must add one scene involving a never-meeting pair.

First, find the mutual characters for your chosen pair — they tell you what world the two characters share.

In [ ]:
# Build cumulative Books 1-5 network (for finding mutual characters)
t0 = time.time()

dfs = [pd.read_csv(DATA + f'hp_book{i}_edges.csv') for i in range(1, 6)]
df_all5 = pd.concat(dfs).groupby(['source', 'target'])['weight'].sum().reset_index()
G_all5 = nx.from_pandas_edgelist(df_all5, 'source', 'target', edge_attr='weight')

print(f'Cumulative Books 1-5: {G_all5.number_of_nodes()} characters, {G_all5.number_of_edges()} pairs | {time.time()-t0:.2f}s')

In [ ]:
# Change these to your chosen pair
char_a = 'Minerva McGonagall'
char_b = 'Cho Chang'

mutual = list(nx.common_neighbors(G_all5, char_a, char_b))

# Sort by degree — most prominent mutual characters first
mutual_sorted = sorted(mutual, key=lambda n: G_all5.degree(n, weight='weight'), reverse=True)

print(f'{char_a} and {char_b} share {len(mutual)} mutual characters:\n')
print(', '.join(mutual_sorted[:15]))

**Your 60-second screenwriter pitch:**

**Pair:** 

**CN score:** 

**3 mutual characters (the world they share):**
1. 
2. 
3. 

**The scene** — where, what happens, what do they say:

*Write here.*

**Why Rowling kept them apart:**

*Write here.*

**Why a film would add it:**

*Write here.*

---
## What to bring to the presentation session

From all three notebooks:

1. Your network expansion paragraph (from Notebook 1)
2. Your community naming tables (from Notebook 2)
3. Your Hermione > Ron argument (from Notebook 2)
4. Your screenwriter pitch (from this notebook)
5. Your sealed predictions — **open them now.** Compare to what you found.

**Final reflection — write 3–4 sentences:** What did you learn from this project that you didn't know from reading? Reference at least one specific number from the data.

*Write here.*